In [ ]:
import os
import json

import Enrich_Job_Offers_Dataframes as de
import Silver_Incremental as inc

# ---------------------------------------------------------------------------
# Configuracion externalizada (widgets de Databricks / variables de entorno).
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "job_offers")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("lookback_days", "7")
dbutils.widgets.text("from_date", "")
dbutils.widgets.text("to_date", "")
dbutils.widgets.text("force_full", "false")
dbutils.widgets.text("repeat_window", "false")


def _cfg(name: str, default: str = "") -> str:
    """Resuelve un parametro: widget -> variable de entorno -> default."""
    value = dbutils.widgets.get(name)
    return value if value else os.environ.get(name.upper(), default)


CATALOG = _cfg("catalog", "job_offers")
BRONZE_SCHEMA = _cfg("bronze_schema", "bronze")
SILVER_SCHEMA = _cfg("silver_schema", "silver")

# Contrato de Silver_Incremental.run: ventana (watermark y backfill) y repeticion.
LOOKBACK_DAYS = _cfg("lookback_days", "7")
FROM_DATE = _cfg("from_date")
TO_DATE = _cfg("to_date")
FORCE_FULL = _cfg("force_full", "false")
REPEAT_WINDOW = _cfg("repeat_window", "false")


def bronze_table(name: str) -> str:
    return f"{CATALOG}.{BRONZE_SCHEMA}.{name}"


def silver_table(name: str) -> str:
    return f"{CATALOG}.{SILVER_SCHEMA}.{name}"

Función enrich pobla las columnas skills y experience_level a través de la columna 'description_clean' (si ya existe un valor no lo sobreescribe).
La columna employment_type trata de poblarla mediante el mismo campo también.
Para agrupar mejor los distintos roles que se ofertan se genera la columna role_category a través de una lista de keywords.
Por último, se normaliza el salario para poder comparar distintos trabajos con salario anual/mensual/por hora.

Normalize Company_Name using 'Title Case' by word

Function to drop duplicates by Job_Title, Company_Name and Location_City

In [ ]:
import pyspark.sql.types as T


df_structure_Schema=T.StructType([
T.StructField("job_id", T.StringType(), True),
T.StructField("job_url", T.StringType(), True),
T.StructField("title", T.StringType(), True),
T.StructField("company_name", T.StringType(), True),
T.StructField("location_city", T.StringType(), True),
T.StructField("location_region", T.StringType(), True),
T.StructField("location_country", T.StringType(), True),
T.StructField("posted_date", T.DateType(), True),
T.StructField("work_mode", T.StringType(), True),
T.StructField("is_salary_available", T.BooleanType(), True),
T.StructField("salary_currency", T.StringType(), True),
T.StructField("salary_min", T.DoubleType(), True),
T.StructField("salary_max", T.DoubleType(), True),
T.StructField("salary_period", T.StringType(), True),
T.StructField("salary_raw", T.StringType(), True),
T.StructField("experience_level", T.StringType(), True),
T.StructField("description_clean", T.StringType(), True),
T.StructField("skills", T.StringType(), True),
T.StructField("source_scraper", T.StringType(), True),
T.StructField("search_role", T.StringType(), True),
T.StructField("scraped_at", T.TimestampType(), True),
T.StructField("_ingest_date", T.DateType(), True)])


from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

def normalize_company_name(col: Column) -> Column:
    """Title case por palabra + trim de espacios sobrantes."""
    return F.initcap(F.trim(F.regexp_replace(col, r"\s+", " ")))

def apply_company_normalization(df: DataFrame, col_name: str = "company_name") -> DataFrame:
    """Devuelve un nuevo DataFrame con la columna indicada normalizada."""
    return df.withColumn(col_name, normalize_company_name(F.col(col_name)))



# --- Agrupacion de las empresas mas grandes (normalizacion canonica) ---
# Las variantes de una misma compania ("Google" / "Google Ireland Ltd" /
# "Google DeepMind") se agrupan bajo un UNICO nombre para que el conteo de
# ofertas por empresa agregue correctamente en BI.
import re
from pyspark.sql import functions as F

COMPANY_ALIAS_PATTERNS = [
    # --- Big tech ---
    ("Google", r"(?i)\bgoogle\b"),
    ("Amazon", r"(?i)\bamazon\b|\baws\b"),
    ("Microsoft", r"(?i)\bmicrosoft\b"),
    ("Apple", r"(?i)\bapple\b"),
    ("Meta", r"(?i)\bmeta\b"),
    ("IBM", r"(?i)\bibm\b"),
    ("Salesforce", r"(?i)\bsalesforce\b"),
    ("Oracle", r"(?i)\boracle\b"),
    ("Nvidia", r"(?i)\bnvidia\b"),
    ("Intel", r"(?i)\bintel\b"),
    ("Samsung", r"(?i)\bsamsung\b"),
    ("SAP", r"(?i)\bsap\b"),
    ("Logitech", r"(?i)\blogitech\b"),
    ("Nokia", r"(?i)\bnokia\b"),
    ("Lenovo", r"(?i)\blenovo\b"),
    # --- Consultoras y servicios TI ---
    ("Accenture", r"(?i)\baccenture\b"),
    ("Deloitte", r"(?i)\bdeloitte\b"),
    ("Capgemini", r"(?i)\bcapgemini\b"),
    ("Cognizant", r"(?i)\bcognizant\b"),
    ("EPAM", r"(?i)\bepam\b"),
    ("Infosys", r"(?i)\binfosys\b"),
    ("Globant", r"(?i)\bglobant\b"),
    ("McKinsey", r"(?i)\bmckinsey\b|\bquantumblack\b"),
    ("EY", r"(?i)\bey\b|\bernst\s*&\s*young\b"),
    ("PwC", r"(?i)\bpwc\b"),
    ("KPMG", r"(?i)\bkpmg\b"),
    ("UST", r"(?i)\bust\b"),
    ("NTT Data", r"(?i)\bntt\s+data\b"),
    ("Sopra Steria", r"(?i)\bsopra\b"),
    ("Inetum", r"(?i)\binetum\b"),
    ("Devoteam", r"(?i)\bdevoteam\b"),
    ("Adecco", r"(?i)\badecco\b"),
    ("CGI", r"(?i)\bcgi\b"),
    ("Logicalis", r"(?i)\blogicalis\b"),
    ("Aubay", r"(?i)\baubay\b"),
    ("Indra", r"(?i)\bindra\b"),
    ("Izertis", r"(?i)\bizertis\b"),
    ("Babel", r"(?i)\bbabel\b"),
    ("Manpower", r"(?i)\bmanpower(?:group)?\b"),
    ("Randstad", r"(?i)\brandstad\b"),
    ("Gi Group", r"(?i)\bgi\s+group\b"),
    ("Hays", r"(?i)\bhays\b"),
    ("Teleperformance", r"(?i)\bteleperformance\b"),
    ("Acciona", r"(?i)\bacciona\b"),
    ("Criteo", r"(?i)\bcriteo\b"),
    ("Adevinta", r"(?i)\badevinta\b"),
    # --- Banca, seguros y servicios financieros ---
    ("JPMorgan Chase", r"(?i)\bjp\s*morgan(?:chase)?\b"),
    ("Morgan Stanley", r"(?i)\bmorgan\s+stanley\b"),
    ("Mastercard", r"(?i)\bmastercard\b"),
    ("Santander", r"(?i)\bsantander\b"),
    ("BBVA", r"(?i)\bbbva\b"),
    ("CaixaBank", r"(?i)\bcaixabank\b|\bvidacaixa\b"),
    ("ABN AMRO", r"(?i)\babn\s+amro\b"),
    ("Rabobank", r"(?i)\brabobank\b"),
    ("Deutsche Bank", r"(?i)\bdeutsche\s+bank\b"),
    ("Deutsche Telekom", r"(?i)\bdeutsche\s+telekom\b"),
    ("Zurich", r"(?i)\bzurich\b"),
    ("Northern Trust", r"(?i)\bnorthern\s+trust\b"),
    ("Marsh", r"(?i)\bmarsh\b"),
    ("UPMC", r"(?i)\bupmc\b"),
    ("TD SYNNEX", r"(?i)\btd\s+synnex\b"),
    # --- Telecom, transporte y energia ---
    ("Vodafone", r"(?i)\bvodafone(?:ziggo)?\b"),
    ("Telefonica", r"(?i)\btelef[oó]nica\b"),
    ("Orange", r"(?i)\borange\b(?!\s+quarter)"),
    ("T-Mobile", r"(?i)\bt[- ]?mobile\b"),
    ("Verizon", r"(?i)\bverizon\b"),
    ("Boeing", r"(?i)\bboeing\b"),
    ("Airbus", r"(?i)\bairbus\b"),
    # --- Industrial y hardware ---
    ("Siemens", r"(?i)\bsiemens\b"),
    ("Bosch", r"(?i)\bbosch\b"),
    ("ABB", r"(?i)\babb\b"),
    ("Abbott", r"(?i)\babbott\b"),
    ("Philips", r"(?i)\bphilips\b"),
    ("Lockheed Martin", r"(?i)\blockheed\b"),
    ("Northrop Grumman", r"(?i)\bnorthrop\b"),
    ("Honeywell", r"(?i)\bhoneywell\b"),
    ("Vertiv", r"(?i)\bvertiv\b"),
    ("Stryker", r"(?i)\bstryker\b"),
    ("Analog Devices", r"(?i)\banalog\s+devices\b"),
    ("Bentley Systems", r"(?i)\bbentley\s+systems\b"),
    ("Keysight", r"(?i)\bkeysight\b"),
    ("Keyrus", r"(?i)\bkeyrus\b"),
    ("Keyrock", r"(?i)\bkeyrock\b"),
    ("Disney", r"(?i)\bdisney\b"),
    # --- Farma y consumo ---
    ("Roche", r"(?i)\broche\b"),
    ("Thermo Fisher", r"(?i)\bthermo\s+fisher\b"),
    ("Novartis", r"(?i)\bnovartis\b"),
    ("AstraZeneca", r"(?i)\bastrazeneca\b"),
    ("Pfizer", r"(?i)\bpfizer\b"),
    ("Sanofi", r"(?i)\bsanofi\b"),
    ("Merck", r"(?i)\bmerck\b"),
    ("Bayer", r"(?i)\bbayer\b"),
    ("Johnson & Johnson", r"(?i)\bjohnson\s*&\s*johnson\b|\bjanssen\b"),
    ("Bristol Myers Squibb", r"(?i)\bbristol\b"),
    ("Eli Lilly", r"(?i)\blilly\b"),
    ("Takeda", r"(?i)\btakeda\b"),
    ("Medtronic", r"(?i)\bmedtronic\b"),
    ("PepsiCo", r"(?i)\bpepsico\b"),
    ("Procter & Gamble", r"(?i)\bprocter\b"),
    ("Nestle", r"(?i)\bnestl[ée]\b"),
    ("Unilever", r"(?i)\bunilever\b"),
    ("Danone", r"(?i)\bdanone\b"),
    ("Coca-Cola", r"(?i)\bcoca\s*-?\s*cola\b"),
    ("Heineken", r"(?i)\bheineken\b"),
    ("Hershey", r"(?i)\bhershey\b"),
    ("Philip Morris", r"(?i)\bphilip\s+morris\b"),
    ("Nike", r"(?i)\bnike\b"),
    ("Adidas", r"(?i)\badidas\b"),
    # --- Internet, SaaS y retail ---
    ("Fever", r"(?i)\bfever(?:up)?\b"),
    ("Glovo", r"(?i)\bglovo\b"),
    ("Revolut", r"(?i)\brevolut\b(?!\s+f1)"),
    ("Booking", r"(?i)\bbooking\b"),
    ("Expedia", r"(?i)\bexpedia\b"),
    ("Adyen", r"(?i)\badyen\b"),
    ("Stripe", r"(?i)\bstripe\b"),
    ("Twilio", r"(?i)\btwilio\b"),
    ("Databricks", r"(?i)\bdatabricks\b"),
    ("Snowflake", r"(?i)\bsnowflake\b"),
    ("ServiceNow", r"(?i)\bservicenow\b"),
    ("Workday", r"(?i)\bworkday\b"),
    ("GitHub", r"(?i)\bgithub\b"),
    ("CrowdStrike", r"(?i)\bcrowdstrike\b"),
    ("Uber", r"(?i)\buber\b"),
    ("Carrefour", r"(?i)\bcarrefour\b"),
    ("Lidl", r"(?i)\blidl\b"),
    ("IKEA", r"(?i)\bikea\b"),
    ("Just Eat Takeaway", r"(?i)\bjust\s+eat\b"),
    ("Zalando", r"(?i)\bzalando\b"),
]


def company_name_canonical(name: str) -> str:
    """Nombre canonico de empresa (first-match wins)."""
    value = (name or "").strip()
    for canon, pattern in COMPANY_ALIAS_PATTERNS:
        if re.search(pattern, value):
            return canon
    return value

def apply_company_canonical(df, col_name: str = "company_name"):
    """Aplica la normalizacion canonica de empresas (columnar)."""
    from pyspark.sql import functions as F
    base = F.trim(F.coalesce(F.col(col_name), F.lit("")))
    expr = base
    for canon, pattern in COMPANY_ALIAS_PATTERNS:
        expr = F.when(base.rlike(pattern), F.lit(canon)).otherwise(expr)
    return df.withColumn(col_name, expr)


def prepare_bronze(df):
    from pyspark.sql.functions import when,col
    from pyspark.sql.functions import lit

    df=df.withColumnsRenamed({
    "id_oferta":"job_id",
    "url_oferta":"job_url",
    "titulo":"title",
    "empresa":"company_name",
    "ciudad":"location_city",
    "provincia":"location_region",
    "pais":"location_country",
    "fecha_publicacion":"posted_date",
    "modalidad":"work_mode",
    "moneda":"salary_currency",
    "salario_min":"salary_min",
    "salario_max":"salary_max",
    "salario_raw":"salary_raw",
    "periodo":"salary_period",
    "experience_level":"experience_level",
    "descripcion_snippet":"description_clean",
    "jornada":"employment_type",
    "fecha_scraped":"scraped_at",
    "fuente":"source_scraper",
    "keyword_buscada":"search_role"
    })

    df=df.withColumns({"is_salary_available":when(col('salary_raw') != "", True).otherwise(False),
                      "skills":lit(""),
                      "experience_level":lit("")})
    return df


def build_offers(df):
    from pyspark.sql.functions import col
    from pyspark.sql import Window
    from pyspark.sql import functions as F

    df_final = df.select(
        [col(c).cast(df_structure_Schema[c].dataType).alias(c) for c in df_structure_Schema.fieldNames()]
    )

    df_final=de.enrich(df_final, add_location_fill=True, normalize_location_city=True)

    df_final=apply_company_normalization(df_final)

    df_final=apply_company_canonical(df_final)

    w = Window.partitionBy("_key").orderBy(F.desc("posted_date"), F.desc("_ingest_date"))

    df_final = (df_final.withColumn("_key", inc.key_expression())
               .withColumn("_rn", F.row_number().over(w))
               .filter(F.col("_rn") == 1)
               .drop("_key", "_rn"))

    df_final=df_final.drop(*['scraped_at','salary_period','work_mode'])

    df_final=df_final.withColumnsRenamed({"salary_period_norm": "salary_period","work_mode_norm": "work_mode"})
    return df_final


def normalize_company(df):
    """Normalizacion de empresa de la silver (initcap + canonico)."""
    return apply_company_canonical(apply_company_normalization(df))

In [ ]:
df = prepare_bronze(spark.read.table(bronze_table("infojobs")))

In [ ]:
summary = inc.run(
    spark,
    silver_table("offers_infojobs"),
    df,
    source="infojobs",
    transform=build_offers,
    normalize_company=normalize_company,
    normalize_city_column=de.location_city_normalized_column,
    lookback_days=LOOKBACK_DAYS,
    from_date=FROM_DATE,
    to_date=TO_DATE,
    force_full=FORCE_FULL,
    repeat_window=REPEAT_WINDOW,
)

In [ ]:
companies_df = (spark.read.table(silver_table("offers_infojobs"))
                .select("company_name").dropDuplicates())
companies_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table("companies_infojobs"))

In [ ]:
dbutils.notebook.exit(json.dumps(summary, ensure_ascii=True,
                                 separators=(",", ":"), default=str))